# 🔍 TruthScan — Layer 1 Training
## Fine-tune DistilBERT on the LIAR dataset

This notebook trains the text-credibility classifier used by TruthScan Layer 1.

- **Model:** `distilbert-base-uncased` → fine-tuned to 3-class credibility
- **Dataset:** [LIAR](https://huggingface.co/datasets/liar) (12.8k labeled statements)
- **Output:** Saved checkpoint in `./models/liar_distilbert/`

**Typical training time:**
- GPU (T4 on Colab) → ~8-12 minutes
- CPU → ~3-4 hours (not recommended)

> **Colab users:** Make sure to set the runtime to GPU: `Runtime → Change runtime type → T4 GPU`

## 1. Install Dependencies
Run this cell if you're on Google Colab (packages are pre-installed locally).

In [ ]:
# Uncomment the line below if running on Google Colab
# !pip install torch transformers accelerate datasets sentencepiece -q

## 2. Imports & Configuration

In [ ]:
import os
import torch
import numpy as np
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from datasets import load_dataset
from torch.utils.data import Dataset as TorchDataset

# ── Config ──────────────────────────────────────────────
MODEL_NAME = "distilbert-base-uncased"
MODEL_DIR  = "./models/liar_distilbert"
EPOCHS     = 3
BATCH_SIZE = 32

# LIAR's 6 classes → 3-class credibility buckets
# 0=pants-fire, 1=false, 2=barely-true, 3=half-true, 4=mostly-true, 5=true
LIAR_INT_MAP = {
    0: 0,  # False
    1: 0,  # False
    2: 1,  # Uncertain
    3: 1,  # Uncertain
    4: 2,  # Credible
    5: 2,  # Credible
}

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 3. Load LIAR Dataset

In [ ]:
print("⏳ Loading LIAR dataset from HuggingFace…")
raw = load_dataset("liar")
print(f"  Train:      {len(raw['train']):,} examples")
print(f"  Validation: {len(raw['validation']):,} examples")
print(f"  Test:       {len(raw['test']):,} examples")

# Preview a few examples
for i in range(3):
    item = raw["train"][i]
    print(f"\n  [{i}] label={item['label']} → {LIAR_INT_MAP[item['label']]}")
    print(f"      {item['statement'][:100]}")

## 4. Prepare Tokenized Dataset

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)


class LiarDataset(TorchDataset):
    def __init__(self, split):
        self.data = raw[split]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        enc = tokenizer(
            item["statement"],
            truncation=True,
            max_length=256,
            padding="max_length",
        )
        return {
            **{k: torch.tensor(v) for k, v in enc.items()},
            "labels": torch.tensor(LIAR_INT_MAP[item["label"]], dtype=torch.long),
        }


train_ds = LiarDataset("train")
val_ds   = LiarDataset("validation")
test_ds  = LiarDataset("test")

print(f"✅ Datasets ready — train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")

## 5. Train DistilBERT

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3
)

args = TrainingArguments(
    output_dir=MODEL_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=64,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_steps=100,
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
)

print("🚀 Training DistilBERT on LIAR…")
trainer.train()

## 6. Save Model

In [ ]:
trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)
print(f"✅ Model saved to {MODEL_DIR}")
print(f"   Files: {os.listdir(MODEL_DIR)}")

## 7. Quick Evaluation

In [ ]:
results = trainer.evaluate(test_ds)
print(f"\n📊 Test set results:")
for k, v in results.items():
    print(f"   {k}: {v:.4f}" if isinstance(v, float) else f"   {k}: {v}")

## 8. Smoke Test — Try Some Predictions

In [ ]:
ID2CRED = {
    0: "❌ Likely False",
    1: "⚠️ Uncertain",
    2: "✅ Likely Credible",
}

test_claims = [
    "The moon landing was faked by NASA in 1969.",
    "Water boils at 100 degrees Celsius at sea level.",
    "5G towers cause COVID-19 infections.",
    "The Earth orbits the Sun once every 365.25 days.",
    "Eating carrots gives you night vision.",
]

model.eval()
model.to(device)

print("🧪 Smoke test predictions:\n")
for claim in test_claims:
    inputs = tokenizer(claim, return_tensors="pt", truncation=True, max_length=256, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1).squeeze().cpu().numpy()
    pred  = int(np.argmax(probs))
    conf  = float(probs[pred])
    print(f"  {ID2CRED[pred]} ({conf:.1%})  →  {claim}")

---
## Next Steps

1. **Download the model folder** (`./models/liar_distilbert/`) from the Colab file browser
2. Place it in your local TruthScan project at the same path
3. Run `python app.py` to start the full analysis pipeline

> The model is ~260 MB. For HuggingFace Spaces deployment, commit it with Git LFS.